# Particle Filter — Non-Gaussian Process Noise

## Overview

This notebook demonstrates the SIR particle filter on a **manoeuvring target with non-Gaussian process noise**. Every filter in the Kalman family (KF, EKF, UKF) propagates uncertainty through the predict step as a single Gaussian — even the UKF, which only approximates the *shape* of a Gaussian more accurately. When the true process noise is non-Gaussian, this structural assumption breaks regardless of how sophisticated the linearisation is.

### Why this scenario requires a Particle Filter

The target moves with approximate constant velocity but is subject to **occasional sudden manoeuvres**: at each timestep, with probability $p_m = 0.15$, the acceleration perturbation is drawn from a wide Gaussian ($\sigma_\text{big} = 3\ \text{m/s}^2$) rather than the narrow baseline ($\sigma_\text{small} = 0.2\ \text{m/s}^2$). This **mixture-of-Gaussians (GMM) process noise** makes the one-step predictive distribution non-Gaussian — it has a sharp central peak (normal sailing) with heavy tails (manoeuvre events).

A filter forced to represent the predictive distribution as a single Gaussian must commit to one $Q$ matrix:
- **Use $Q_\text{small}$** — tracks accurately during normal sailing but cannot absorb manoeuvres; innovations spike and the filter loses the target.
- **Use $Q_\text{big}$** — survives manoeuvres but is permanently diffuse, inflating estimation error during normal sailing.

The particle filter sidesteps this entirely. Each particle independently samples from the GMM at every predict step, so the particle cloud naturally maintains hypotheses under **both** the normal and manoeuvre modes simultaneously — without committing to either.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D

np.random.seed(42)

## Mathematical Setup

### State and Process Model

State vector: $\mathbf{x} = [p_x,\ p_y,\ v_x,\ v_y]^\top$. Constant-velocity dynamics with GMM process noise:

$$
\mathbf{x}_{k+1} = F\mathbf{x}_k + \mathbf{w}_k, \qquad
F = \begin{bmatrix} 1 & 0 & \Delta t & 0 \\ 0 & 1 & 0 & \Delta t \\ 0 & 0 & 1 & 0 \\ 0 & 0 & 0 & 1 \end{bmatrix}
$$

The process noise $\mathbf{w}_k$ is drawn from a two-component mixture:

$$
\mathbf{w}_k \sim \begin{cases}
  \mathcal{N}(\mathbf{0},\ \sigma_\text{small}^2\, I_2) & \text{with probability}\ 1 - p_m \quad \text{(normal sailing)}\\
  \mathcal{N}(\mathbf{0},\ \sigma_\text{big}^2\,   I_2) & \text{with probability}\ p_m \quad\quad\;\; \text{(manoeuvre)}
\end{cases}
$$

The marginal distribution of $\mathbf{w}_k$ is non-Gaussian: it has a sharp central peak for normal motion and heavy tails for manoeuvre events. No single Gaussian can represent this without loss.

### Measurement Model

A fixed sensor at the origin measures **range and bearing**:

$$
\mathbf{z}_k = \begin{bmatrix} \sqrt{p_x^2 + p_y^2} \\ \arctan2(p_y,\ p_x) \end{bmatrix} + \boldsymbol{\nu}_k,
\qquad \boldsymbol{\nu}_k \sim \mathcal{N}\!\left(\mathbf{0},\ R\right)
$$

This is nonlinear but Gaussian — the scenario is standard for EKF/UKF. The non-Gaussian challenge is entirely in the **process model**.

### SIR Particle Filter

Each particle $\mathbf{x}_k^{(i)}$ represents one hypothesis about the state. At each step:

1. **Predict** — propagate through dynamics, sampling each particle's noise independently from the GMM:
$$\mathbf{x}_k^{(i)} \leftarrow F\mathbf{x}_{k-1}^{(i)} + \mathbf{w}_k^{(i)}, \quad \mathbf{w}_k^{(i)} \sim \text{GMM}$$
Unlike a Kalman filter, there is no $Q$ matrix: the mixture is sampled directly.

2. **Update** — weight by likelihood:
$$w_k^{(i)} \propto w_{k-1}^{(i)} \cdot p(\mathbf{z}_k \mid \mathbf{x}_k^{(i)})$$

3. **Resample** — when $N_\text{eff} = \bigl(\sum_i (w_k^{(i)})^2\bigr)^{-1} < N/2$, draw $N$ new particles by **systematic resampling** (lower variance than multinomial).

In [ ]:
# ============================================================
#  Simulation
# ============================================================

def simulate_gmm_cv(x0, N, dt, p_maneuver, sigma_small, sigma_big):
    """
    Simulate CV trajectory with mixture-of-Gaussians process noise.
    State: [px, py, vx, vy].
    Returns states (N, 4) and a boolean maneuver_steps array.
    """
    states = np.zeros((N, 4))
    states[0] = x0
    maneuver_steps = np.zeros(N, dtype=bool)
    for k in range(N - 1):
        is_maneuver = np.random.rand() < p_maneuver
        sigma = sigma_big if is_maneuver else sigma_small
        ax = np.random.randn() * sigma
        ay = np.random.randn() * sigma
        states[k+1, 0] = states[k, 0] + states[k, 2] * dt + 0.5 * ax * dt**2
        states[k+1, 1] = states[k, 1] + states[k, 3] * dt + 0.5 * ay * dt**2
        states[k+1, 2] = states[k, 2] + ax * dt
        states[k+1, 3] = states[k, 3] + ay * dt
        maneuver_steps[k+1] = is_maneuver
    return states, maneuver_steps


def h_rb(x):
    """Range-bearing measurement. State: [px, py, vx, vy]."""
    return np.array([np.hypot(x[0], x[1]), np.arctan2(x[1], x[0])])


def simulate_rb_measurements(states, sigma_r, sigma_theta):
    """Add independent Gaussian noise to range-bearing measurements."""
    N = len(states)
    z = np.zeros((N, 2))
    for k in range(N):
        z[k] = h_rb(states[k])
        z[k, 0] += np.random.randn() * sigma_r
        z[k, 1] += np.random.randn() * sigma_theta
    return z

## Particle Filter Implementation

In [ ]:
# ============================================================
#  Particle filter
# ============================================================

def pf_init(N_particles, x0_mean, x0_std):
    """
    Initialise particles as independent Gaussian draws.
    x0_mean : [px, py, vx, vy] — initial mean estimate
    x0_std  : per-dimension standard deviation (length 4)
    """
    particles = x0_mean + np.random.randn(N_particles, 4) * x0_std
    weights   = np.ones(N_particles) / N_particles
    return particles, weights


def pf_predict(particles, dt, p_maneuver, sigma_small, sigma_big):
    """
    Propagate particles with CV dynamics and GMM process noise.
    Each particle independently draws from the noise mixture:
      - probability p_maneuver : wide noise (sigma_big)
      - probability 1-p_maneuver: narrow noise (sigma_small)
    No Q matrix is formed — the mixture is sampled directly.
    """
    N           = len(particles)
    is_maneuver = np.random.rand(N) < p_maneuver
    sigma_a     = np.where(is_maneuver, sigma_big, sigma_small)
    ax = np.random.randn(N) * sigma_a
    ay = np.random.randn(N) * sigma_a

    new_p = particles.copy()
    new_p[:, 0] = particles[:, 0] + particles[:, 2] * dt + 0.5 * ax * dt**2
    new_p[:, 1] = particles[:, 1] + particles[:, 3] * dt + 0.5 * ay * dt**2
    new_p[:, 2] = particles[:, 2] + ax * dt
    new_p[:, 3] = particles[:, 3] + ay * dt
    return new_p


def pf_update(particles, weights, z, R):
    """
    Reweight particles by Gaussian range-bearing likelihood.
    Log-domain arithmetic avoids underflow with many particles.
    """
    r_pred     = np.hypot(particles[:, 0], particles[:, 1])
    theta_pred = np.arctan2(particles[:, 1], particles[:, 0])

    innov_r     = z[0] - r_pred
    innov_theta = z[1] - theta_pred
    innov_theta = (innov_theta + np.pi) % (2 * np.pi) - np.pi  # wrap to [-pi, pi]

    log_like  = -0.5 * (innov_r**2 / R[0, 0] + innov_theta**2 / R[1, 1])
    log_like -= log_like.max()   # subtract max before exp for numerical stability
    weights   = weights * np.exp(log_like)

    total = weights.sum()
    if total < 1e-300:           # all particles collapsed — reset uniformly
        weights[:] = 1.0 / len(weights)
    else:
        weights /= total
    return weights


def effective_sample_size(weights):
    """N_eff = 1 / sum(w^2). Ranges from 1 (total degeneracy) to N (uniform)."""
    return 1.0 / np.sum(weights**2)


def systematic_resample(weights):
    """
    Systematic resampling: O(N), lower variance than multinomial.
    A single uniform draw u ~ U[0, 1/N] offsets a regular grid of N positions
    across the CDF, ensuring each stratum gets exactly one sample.
    """
    N         = len(weights)
    positions = (np.arange(N) + np.random.uniform(0, 1)) / N
    cumsum    = np.cumsum(weights)
    return np.searchsorted(cumsum, positions)


def pf_resample(particles, weights, threshold=0.5):
    """Resample when N_eff < threshold * N to combat weight degeneracy."""
    N = len(weights)
    if effective_sample_size(weights) < threshold * N:
        indices   = systematic_resample(weights)
        particles = particles[indices]
        weights   = np.ones(N) / N
    return particles, weights


def pf_estimate(particles, weights):
    """Posterior mean: weighted average of the particle cloud."""
    return np.sum(particles * weights[:, np.newaxis], axis=0)

---
## Scenario

A target starts east of the sensor and moves north-east at moderate speed. Manoeuvres occur randomly at 15% of timesteps — sharp velocity kicks that no single Gaussian $Q$ can anticipate.

| Parameter | Value | Note |
|---|---|---|
| $\mathbf{x}_0^\text{true}$ | $[400, 300, 5, 2]^\top$ | px (m), py (m), vx (m/s), vy (m/s) |
| $\Delta t$ | 1.0 s | |
| $N$ | 200 steps | |
| $p_m$ | 0.15 | manoeuvre probability per step |
| $\sigma_\text{small}$ | 0.2 m/s² | normal sailing noise |
| $\sigma_\text{big}$ | 3.0 m/s² | manoeuvre noise |
| $\sigma_r$ | 15 m | range measurement noise |
| $\sigma_\theta$ | 1.5° | bearing measurement noise |
| $N_\text{particles}$ | 2000 | |

In [ ]:
# ============================================================
#  Scenario setup
# ============================================================

dt          = 1.0
N           = 200
N_particles = 2000

p_maneuver  = 0.15
sigma_small = 0.2    # m/s^2 — normal sailing
sigma_big   = 3.0    # m/s^2 — manoeuvre

sigma_r     = 15.0
sigma_theta = np.deg2rad(1.5)
R = np.diag([sigma_r**2, sigma_theta**2])

x0_true = np.array([400.0, 300.0, 5.0, 2.0])

# Simulate truth and measurements
true_states, maneuver_steps = simulate_gmm_cv(
    x0_true, N, dt, p_maneuver, sigma_small, sigma_big
)
z_meas = simulate_rb_measurements(true_states, sigma_r, sigma_theta)

print(f"Total manoeuvre steps: {maneuver_steps.sum()} / {N} "
      f"({100 * maneuver_steps.mean():.0f}% — expected ~{100*p_maneuver:.0f}%)")

# Select snapshot steps: anchor two around manoeuvre events for the cloud plots
useful_m = np.where(maneuver_steps & (np.arange(N) > 5) & (np.arange(N) < N - 5))[0]
if len(useful_m) >= 2:
    m1 = int(useful_m[len(useful_m) // 4])
    m2 = int(useful_m[3 * len(useful_m) // 4])
else:
    m1, m2 = N // 3, 2 * N // 3
snapshot_steps = sorted(set([
    5,
    m1, min(m1 + 5, N - 1),
    m2, min(m2 + 5, N - 1),
    N - 1
]))[:6]
print(f"Snapshot steps: {snapshot_steps}")
print(f"  (steps {m1} and {m2} are manoeuvre events)")

# Initialise PF from first measurement (range + bearing → rough Cartesian estimate)
r0, th0 = z_meas[0]
x0_pf   = np.array([r0 * np.cos(th0), r0 * np.sin(th0), 0.0, 0.0])
x0_std  = np.array([sigma_r, sigma_r, 5.0, 5.0])   # velocity prior: ±5 m/s
particles, weights = pf_init(N_particles, x0_pf, x0_std)

In [ ]:
# ============================================================
#  Run particle filter
# ============================================================

estimates  = np.zeros((N, 4))
n_eff_log  = np.zeros(N)
snapshots  = {}   # step -> (particles copy, weights copy)

for k in range(N):
    particles = pf_predict(particles, dt, p_maneuver, sigma_small, sigma_big)
    weights   = pf_update(particles, weights, z_meas[k], R)
    n_eff_log[k] = effective_sample_size(weights)

    if k in snapshot_steps:
        snapshots[k] = (particles.copy(), weights.copy())

    particles, weights = pf_resample(particles, weights)
    estimates[k] = pf_estimate(particles, weights)

# Metrics
pos_err = np.sqrt(
    (estimates[:, 0] - true_states[:, 0])**2 +
    (estimates[:, 1] - true_states[:, 1])**2
)
rmse = np.sqrt(np.mean(pos_err**2))

r_true      = np.hypot(true_states[:, 0], true_states[:, 1])
noise_floor = np.sqrt(sigma_r**2 + (r_true * sigma_theta)**2)

print(f"Position RMSE     : {rmse:.1f} m")
print(f"Noise floor (mean): {noise_floor.mean():.1f} m")
print(f"Mean N_eff        : {n_eff_log.mean():.0f} / {N_particles}")

In [ ]:
# ============================================================
#  Plot 1: Trajectory + Position Error
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── LEFT: Trajectory ─────────────────────────────────────────────────────────
ax = axes[0]

pts  = true_states[:, :2].reshape(-1, 1, 2)
segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
lc   = LineCollection(segs, cmap='plasma', norm=plt.Normalize(0, N),
                      linewidth=2.5, zorder=6)
lc.set_array(np.arange(N - 1))
ax.add_collection(lc)
cbar = fig.colorbar(lc, ax=ax, pad=0.02)
cbar.set_label('Time step', fontsize=16)
cbar.ax.tick_params(labelsize=15)

# Measurements: project back to Cartesian
meas_x = z_meas[:, 0] * np.cos(z_meas[:, 1])
meas_y = z_meas[:, 0] * np.sin(z_meas[:, 1])
ax.scatter(meas_x, meas_y, c=np.arange(N), cmap='plasma',
           s=15, alpha=0.45, zorder=4, marker='x', linewidths=1.0)

ax.plot(estimates[:, 0], estimates[:, 1],
        'k--', linewidth=1.5, alpha=0.75, zorder=5)

# Mark manoeuvre events on the true trajectory
m_idx = np.where(maneuver_steps)[0]
ax.scatter(true_states[m_idx, 0], true_states[m_idx, 1],
           s=40, c='red', marker='v', zorder=7, alpha=0.7)

ax.scatter([0], [0], s=300, c='black', marker='^', zorder=8)

ax.set_xlabel('$p_x$ (m)', fontsize=17)
ax.set_ylabel('$p_y$ (m)', fontsize=17)
ax.set_title('Trajectory', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.set_aspect('equal')
ax.grid(True, alpha=0.4)

legend_handles = [
    Line2D([0], [0], color='darkorchid', linewidth=2.5),
    Line2D([0], [0], color='k', linewidth=1.5, linestyle='--', alpha=0.75),
    Line2D([0], [0], marker='^', color='w', markerfacecolor='black',
           markersize=11, linestyle='None'),
    Line2D([0], [0], marker='x', color='grey', markersize=8,
           linestyle='None', alpha=0.7),
    Line2D([0], [0], marker='v', color='red', markersize=7,
           linestyle='None', alpha=0.7),
]
ax.legend(legend_handles,
          ['True trajectory', 'PF estimate', 'Sensor',
           'Measurements (Cartesian)', 'Manoeuvre events'],
          fontsize=13, loc='upper left')

# ── RIGHT: Position error ─────────────────────────────────────────────────────
ax = axes[1]

ax.plot(pos_err, color='steelblue', linewidth=1.8,
        label='PF position error (m)')
ax.plot(noise_floor, color='#d62728', linewidth=1.5, linestyle=':',
        label=r'Single-measurement noise floor '
              r'$\sqrt{\sigma_r^2 + (r\sigma_\theta)^2}$'
              f' (mean {noise_floor.mean():.1f} m)')
ax.axhline(rmse, color='k', linewidth=1.5, linestyle='--',
           label=f'RMSE = {rmse:.1f} m')

# Shade manoeuvre timesteps
for mi in m_idx:
    ax.axvspan(mi - 0.5, mi + 0.5, color='red', alpha=0.15, linewidth=0)

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('Position error (m)', fontsize=17)
ax.set_title('Position Error vs Measurement Noise Floor', fontsize=18)
ax.tick_params(axis='both', labelsize=15)
ax.set_xlim(0, N - 1)
ax.set_ylim(bottom=0)
ax.legend(fontsize=13)
ax.grid(True, alpha=0.4)

plt.suptitle('Particle Filter: CV Target with GMM Process Noise',
             fontsize=20, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Plot 2: Particle Cloud Evolution
#  Shows how the cloud spreads and contracts around manoeuvre events
# ============================================================

# Compute axis bounds from the true trajectory
px_min = true_states[:, 0].min() - 150
px_max = true_states[:, 0].max() + 150
py_min = true_states[:, 1].min() - 150
py_max = true_states[:, 1].max() + 150

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.flatten()

for i, step in enumerate(snapshot_steps):
    ax = axes[i]
    snap_p, snap_w = snapshots[step]

    # Particle cloud (size scaled by weight for visibility)
    max_w = snap_w.max()
    sizes = np.clip(800 * snap_w / (max_w + 1e-300), 0.5, 60)
    ax.scatter(snap_p[:, 0], snap_p[:, 1],
               s=sizes, alpha=0.3, c='steelblue', linewidths=0)

    # True position
    ax.scatter(true_states[step, 0], true_states[step, 1],
               s=180, c='green', marker='*', zorder=6,
               label='True position')

    # PF estimate
    ax.scatter(estimates[step, 0], estimates[step, 1],
               s=100, c='black', marker='D', zorder=7,
               label='PF estimate')

    # Sensor
    ax.scatter([0], [0], s=120, c='black', marker='^', zorder=7)

    is_m = maneuver_steps[step]
    title_str = f'$t = {step}$'
    title_str += '  [MANOEUVRE]' if is_m else ''
    ax.set_title(title_str, fontsize=13,
                 color='red' if is_m else 'black')
    ax.set_xlim(px_min, px_max)
    ax.set_ylim(py_min, py_max)
    ax.set_xlabel('$p_x$ (m)', fontsize=13)
    ax.set_ylabel('$p_y$ (m)', fontsize=13)
    ax.tick_params(labelsize=12)
    ax.grid(True, alpha=0.35)
    if i == 0:
        ax.legend(fontsize=11, loc='upper right')

plt.suptitle('Particle Cloud Evolution — Snapshots at Selected Timesteps\n'
             '(particle size proportional to weight; red title = manoeuvre at that step)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
#  Plot 3: N_eff over time
# ============================================================

fig, ax = plt.subplots(figsize=(13, 5))

# Shade manoeuvre timesteps
for mi in m_idx:
    ax.axvspan(mi - 0.5, mi + 0.5, color='red', alpha=0.2, linewidth=0)

ax.plot(n_eff_log, color='steelblue', linewidth=1.5,
        label='$N_\\mathrm{eff}$')
ax.axhline(0.5 * N_particles, color='grey', linestyle='--', linewidth=1.5,
           label=f'Resample threshold ($N/2 = {N_particles//2}$)')

ax.set_xlabel('Time step', fontsize=17)
ax.set_ylabel('$N_\\mathrm{eff}$', fontsize=17)
ax.set_title(
    f'Effective Sample Size\n'
    f'Mean $N_{{\\mathrm{{eff}}}}$ = {n_eff_log.mean():.0f} / {N_particles}  '
    f'| red bands = manoeuvre events',
    fontsize=15
)
ax.tick_params(axis='both', labelsize=15)
ax.set_xlim(0, N - 1)
ax.set_ylim(0, N_particles * 1.05)
ax.legend(fontsize=14)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## Analysis

**Trajectory tracking.** Despite the unpredictable manoeuvre kicks, the particle filter maintains a close estimate throughout. The position error (right panel of Figure 1) spikes briefly at manoeuvre timesteps and recovers within a handful of steps as new range-bearing measurements constrain the cloud.

**RMSE vs noise floor.** The filter's RMSE falls below the single-measurement noise floor $\sqrt{\sigma_r^2 + (r\sigma_\theta)^2}$, reflecting the information accumulated by fusing measurements over time. The intermittent RMSE spikes at manoeuvre events represent the irreducible cost of sudden, unannounced velocity changes — not a filter deficiency.

**Effective sample size.** $N_\text{eff}$ drops sharply at manoeuvre events (Figure 3). This is the expected signature: after a sudden velocity change, the likelihood concentrates weight on the small fraction of particles that happened to draw a large acceleration noise, causing weight degeneracy. The systematic resampler is triggered ($N_\text{eff} < N/2$) and diversifies the cloud, allowing rapid recovery.

**Particle cloud snapshots.** The cloud spreads around manoeuvre timesteps (particles that drew the large-$\sigma$ component) and contracts back to a tight cluster once several measurements have confirmed the new trajectory (Figure 2). This spread-and-contract cycle is the particle filter's analogue of a Kalman covariance inflating during a manoeuvre and re-shrinking with measurements — except here it is represented in the full non-Gaussian sense, without forcing a single-Gaussian approximation on the heavy-tailed predictive distribution.

**Why a Kalman filter cannot do this.** A UKF or EKF must choose a single $Q$ at design time. Choosing $Q_\text{small}$ keeps estimates tight during normal sailing but produces persistent large innovations after manoeuvres, causing the filter to either diverge or require adaptive-$Q$ heuristics. Choosing $Q_\text{big}$ inflates the covariance permanently, trading normal-sailing accuracy for manoeuvre robustness. The particle filter avoids this trade-off entirely: the GMM noise is sampled particle-by-particle, so the predictive distribution naturally reflects both modes — no design-time compromise required.